In [3]:
import numpy as np
from scipy.optimize import minimize

# --- PARAMETRI FISICI DEL ROBOT ---
m = 5.0        # Massa del robot (kg)
J = 0.2        # Momento d'inerzia (kg*m^2)
b = 0.15       # Semicarreggiata (distanza dal centro alla ruota, metri)

# --- PARAMETRI MPC ---
N = 12          # Orizzonte di predizione
dt = 0.05       # Tempo di campionamento (più piccolo per la dinamica)

# Pesi: [x, y, theta, v, omega]
Q = np.diag([15.0, 15.0, 0.5, 0.1, 0.1])
# Pesi sugli ingressi: [f_R, f_L] (costo energetico della spinta)
R = np.diag([0.01, 0.01])        

# Vincoli sulle spinte delle ruote (in Newton, es: max 10N avanti, -10N indietro)
bounds_u = [(-10.0, 10.0), (-10.0, 10.0)] * N  

# --- MODELLO DI PREDIZIONE DINAMICO ---
def robot_dynamic_step(z, u, dt):
    """Calcola lo stato successivo considerando la dinamica delle forze."""
    x, y, theta, v, omega = z
    f_R, f_L = u
    
    # 1. Calcolo delle accelerazioni dalle forze (F = m*a, T = J*alpha)
    a = (f_R + f_L) / m
    alpha = ((f_R - f_L) * b) / J
    
    # 2. Aggiornamento delle velocità (Eulero in avanti)
    v_next = v + a * dt
    omega_next = omega + alpha * dt
    
    # 3. Aggiornamento delle posizioni (cinematica basata sulle nuove velocità)
    x_next = x + v_next * np.cos(theta) * dt
    y_next = y + v_next * np.sin(theta) * dt
    theta_next = theta + omega_next * dt
    
    return np.array([x_next, y_next, theta_next, v_next, omega_next])

# --- FUNZIONE DI COSTO ---
def cost_function(u_flat, z_current, z_target, N, dt, Q, R):
    u = u_flat.reshape((N, 2))
    cost = 0.0
    z = np.copy(z_current)
    
    for k in range(N):
        z = robot_dynamic_step(z, u[k], dt)
        
        # Errore di stato (vogliamo raggiungere x, y, theta target con velocità v=0, omega=0)
        error = z - z_target
        error[2] = (error[2] + np.pi) % (2 * np.pi) - np.pi  # Normalizza theta
        
        # Costo stato + Costo spinta
        cost += error.T @ Q @ error + u[k].T @ R @ u[k]
        
    return cost

# --- SIMULAZIONE ---
# Stato iniziale: fermo nell'origine [x, y, theta, v, omega]
z_current = np.array([0.0, 0.0, 0.0, 0.0, 0.0])

# Stato obiettivo: posizione desiderata a velocità finale zero
z_target = np.array([1.5, 1.5, np.pi/2, 0.0, 0.0])

u_guess = np.zeros(N * 2)

print(f"Stato Iniziale: {z_current}")
print(f"Target: {z_target}\n")

for step in range(50):
    res = minimize(
        cost_function, 
        u_guess, 
        args=(z_current, z_target, N, dt, Q, R),
        method='SLSQP',
        bounds=bounds_u
    )
    
    u_optimal = res.x.reshape((N, 2))
    u_apply = u_optimal[0]  # Applichiamo la prima coppia di spinte [f_R, f_L]
    
    # Avanzamento del robot reale
    z_current = robot_dynamic_step(z_current, u_apply, dt)
    
    distance = np.linalg.norm(z_current[:2] - z_target[:2])
    print(f"Passo {step+1}: Pos=({z_current[0]:.2f}, {z_current[1]:.2f}) | Vel v={z_current[3]:.2f} m/s | Spinte Ruote: R={u_apply[0]:.1f}N, L={u_apply[1]:.1f}N")
    
    u_guess = np.vstack((u_optimal[1:], np.zeros(2))).flatten()
    
    if distance < 0.05 and abs(z_current[3]) < 0.05:
        print("\nTarget raggiunto e robot fermato con successo!")
        break


Stato Iniziale: [0. 0. 0. 0. 0.]
Target: [1.5        1.5        1.57079633 0.         0.        ]

Passo 1: Pos=(0.01, 0.00) | Vel v=0.15 m/s | Spinte Ruote: R=10.0N, L=5.0N
Passo 2: Pos=(0.02, 0.00) | Vel v=0.29 m/s | Spinte Ruote: R=10.0N, L=3.7N
Passo 3: Pos=(0.04, 0.00) | Vel v=0.42 m/s | Spinte Ruote: R=10.0N, L=3.7N
Passo 4: Pos=(0.07, 0.00) | Vel v=0.57 m/s | Spinte Ruote: R=10.0N, L=4.6N
Passo 5: Pos=(0.11, 0.01) | Vel v=0.73 m/s | Spinte Ruote: R=10.0N, L=6.0N
Passo 6: Pos=(0.15, 0.01) | Vel v=0.90 m/s | Spinte Ruote: R=10.0N, L=7.5N
Passo 7: Pos=(0.21, 0.03) | Vel v=1.09 m/s | Spinte Ruote: R=10.0N, L=8.7N
Passo 8: Pos=(0.27, 0.04) | Vel v=1.28 m/s | Spinte Ruote: R=10.0N, L=9.3N
Passo 9: Pos=(0.34, 0.07) | Vel v=1.48 m/s | Spinte Ruote: R=10.0N, L=9.5N
Passo 10: Pos=(0.41, 0.10) | Vel v=1.68 m/s | Spinte Ruote: R=10.0N, L=9.6N
Passo 11: Pos=(0.50, 0.14) | Vel v=1.87 m/s | Spinte Ruote: R=10.0N, L=9.8N
Passo 12: Pos=(0.59, 0.19) | Vel v=2.07 m/s | Spinte Ruote: R=10.0N, L=10.